# Machine Learning — Bike Sharing

Este notebook apresenta a etapa de modelagem preditiva dos dados de aluguel de bicicletas, com o objetivo de desenvolver modelos capazes de prever a demanda por aluguéis nas próximas 24 horas.

A partir dos padrões identificados na análise exploratória, serão utilizadas variáveis temporais, climáticas e informações sobre o comportamento recente da demanda. Também serão avaliadas diferentes abordagens de modelagem e métricas de desempenho, buscando identificar quais modelos apresentam melhores resultados para o problema de previsão.

In [1]:
import pandas as pd

DATA_PATH = '../data/'
file_name = 'bike_sharing_silver.csv'

df = pd.read_csv(DATA_PATH+file_name, sep=',')
df_eda = df.copy()

df.head()

,instant,season,year,month,hour,holiday,weekday,working_day,weather_sit,temp,feeling_temp,hum,wind_speed,casual,registered,datetime,cnt
0,1,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,2011-01-01 00:00:00,16
1,2,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,2011-01-01 01:00:00,40
2,3,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,2011-01-01 02:00:00,32
3,4,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,2011-01-01 03:00:00,13
4,5,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,2011-01-01 04:00:00,1


In [2]:
TARGET_COLUMN = 'cnt'
NUMERIC_COLUMNS = ['temp', 'feeling_temp', 'hum', 'wind_speed', 'casual', 'registered']
CATEGORICAL_COLUMNS = ['holiday', 'weekday', 'working_day', 'weather_sit']
DATETIME_COLUMN = 'datetime'

# Rolling Features

As rolling features permitem resumir o comportamento recente da demanda em diferentes janelas de tempo. A média e a mediana representam o nível central da demanda, enquanto o desvio-padrão representa sua variabilidade. Já os valores mínimo e máximo indicam os extremos observados dentro da janela.

Foram criadas as rolling features de média, mediana, desvio-padrão, mínimo e máximo para as janelas de 6, 24 e 168 horas. A janela de 6 horas representa o comportamento mais recente, enquanto as janelas de 24 e 168 horas permitem capturar comportamentos relacionados aos padrões diário e semanal identificados na análise da série.

Para evitar vazamento de informação, as estatísticas foram calculadas utilizando somente valores anteriores ao instante que será previsto, sem incluir informações futuras.

In [3]:
windows = [6, 24, 168]

for window in windows:
    mean_column = 'rolling_mean_' + str(window)
    median_column = 'rolling_median_' + str(window)
    std_column = 'rolling_std_' + str(window)
    min_column = 'rolling_min_' + str(window)
    max_column = 'rolling_max_' + str(window)
    
    df[mean_column] = df[TARGET_COLUMN].shift(1).rolling(window=window).mean()
    
    df[median_column] = df[TARGET_COLUMN].shift(1).rolling(window=window).median()
    
    df[std_column] = df[TARGET_COLUMN].shift(1).rolling(window=window).std()
    
    df[min_column] = df[TARGET_COLUMN].shift(1).rolling(window=window).min()
    
    df[max_column] = df[TARGET_COLUMN].shift(1).rolling(window=window).max()

df.shape

(17379, 32)

### Lags

As variáveis de lag permitem utilizar valores anteriores da demanda como informações para a previsão, incorporando ao modelo a dependência temporal existente na série. Diferentemente das rolling features, que resumem um conjunto de observações, cada lag representa o valor da demanda em um ponto específico do passado.

Foram criadas as variáveis `lag_1`, `lag_6`, `lag_24` e `lag_168`. O `lag_1` representa a demanda da hora anterior, enquanto o `lag_6` representa a demanda observada seis horas antes. Já os `lag_24` e `lag_168` permitem utilizar, respectivamente, a demanda do mesmo horário do dia anterior e da semana anterior, considerando os padrões diário e semanal identificados na análise da série.

A criação desses lags foi baseada nos padrões de autocorrelação observados durante a análise exploratória, especialmente nas defasagens relacionadas ao comportamento diário e semanal. Dessa forma, essas variáveis permitem que o modelo utilize informações recentes e padrões recorrentes da demanda para realizar as previsões.

In [4]:
for lag in [1, 6, 24, 168]:
    df[f'lag_{lag}'] = df[TARGET_COLUMN].shift(lag)

In [5]:
df.shape

(17379, 36)